# 01 · Build the evaluation set (COCO val2017)

**Why COCO val2017.** It is the standard *held-out* set: YOLO11n was trained on COCO **train**2017, never on these images, and every object has an exact human-drawn outline and box, i.e. ground truth to measure against.

This notebook, once:
1. downloads COCO's official annotations (one 241 MB zip; only `instances_val2017.json` is kept),
2. picks two **disjoint** image sets with a fixed seed (so anyone re-running gets the same images):
   - **100 images** for the automatic model evaluation (notebook 02),
   - **20 images** for the timed effort study (notebook 03), split into halves **A** and **B** with balanced object counts,
3. downloads just those images (~120 × 160 kB),
4. copies the study images to `eval/data/effort/A` and `eval/data/effort/B`, ready to upload.

The 15 practice images in `data/test_image` are excluded, so nothing you have already labelled is evaluated.

In [5]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "eval" else pathlib.Path.cwd()))
from eval import evallib as E
E.setup()          # work from the project folder, where the model weights are
print("project folder:", E.ROOT)

project folder: C:\Users\yuvra\Desktop\UNI\Ai Vision\AI annotator


In [6]:
coco = E.load_coco()
subsets = E.choose_subsets(coco)
print("model-evaluation images:", len(subsets["model"]))
print("effort study, half A:", len(subsets["effort_a"]), " half B:", len(subsets["effort_b"]))

downloading COCO annotations (241 MB, once) ...
loading annotations into memory...
Done (t=0.59s)
creating index...
index created!
model-evaluation images: 100
effort study, half A: 10  half B: 10


In [7]:
from tqdm.auto import tqdm
for i in tqdm(subsets["model"] + subsets["effort_a"] + subsets["effort_b"], desc="images"):
    E.image_path(i)                       # downloads once, then cached in eval/data/images
folders = E.copy_effort_images(subsets)
folders

c:\Users\yuvra\Desktop\UNI\Ai Vision\AI annotator\.venv311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
images:   0%|          | 0/120 [00:00<?, ?it/s]

downloading http://images.cocodataset.org/val2017/000000251119.jpg ...


images:   1%|          | 1/120 [00:00<01:51,  1.07it/s]

downloading http://images.cocodataset.org/val2017/000000537355.jpg ...


images:   2%|▏         | 2/120 [00:01<01:45,  1.12it/s]

downloading http://images.cocodataset.org/val2017/000000368940.jpg ...


... (232 outputs left out)

images:  99%|█████████▉| 119/120 [01:47<00:00,  1.73it/s]

downloading http://images.cocodataset.org/val2017/000000470773.jpg ...


images: 100%|██████████| 120/120 [01:48<00:00,  1.11it/s]


{'A': WindowsPath('C:/Users/yuvra/Desktop/UNI/Ai Vision/AI annotator/eval/data/effort/A'),
 'B': WindowsPath('C:/Users/yuvra/Desktop/UNI/Ai Vision/AI annotator/eval/data/effort/B')}

In [8]:
# What the timed study asks for: objects of these classes, at least 32x32 px.
cat_ids = coco.getCatIds(catNms=E.EFFORT_CLASSES)
for half in ("a", "b"):
    n = 0
    for i in subsets[f"effort_{half}"]:
        anns = coco.loadAnns(coco.getAnnIds(imgIds=i, catIds=cat_ids, iscrowd=False))
        n += sum(a["area"] >= E.MIN_EFFORT_AREA for a in anns)
    print(f"half {half.upper()}: {len(subsets['effort_' + half])} images, {n} objects to label")

half A: 10 images, 30 objects to label
half B: 10 images, 26 objects to label


**Done when** the last cell prints two halves with similar object counts. Next: notebook 02 (automatic, ~15 min on CPU) and, separately, the timed study described in notebook 03.